# GeoJam 2026 — Data Preparation

This notebook downloads and processes all data needed for the GeoJam hackathon challenge:
**predicting and explaining vegetation change across London using satellite imagery and urban features.**

Run this once before the event. The output is a `geojam_data.zip` file containing:
- Raw Sentinel-2 bands (B02, B03, B04, B08) for two summer scenes ~5 years apart
- Pre-computed NDVI rasters for both years
- MSOA boundaries with non-satellite features (population, density, distance to centre, parks)
- A full escape-hatch GeoDataFrame with satellite-derived features already joined
- Raster metadata so participants can map pixels to coordinates

**No authentication is required for any data source.**

### Sources
- **Satellite imagery:** Sentinel-2 L2A via Microsoft Planetary Computer (STAC API)
- **Boundaries:** MSOA 2021 from ONS Open Geography Portal
- **Population:** Census 2021 from NOMIS
- **Parks:** OpenStreetMap via OSMnx

In [1]:
!pip install -q pystac-client planetary-computer rasterio rasterstats osmnx geopandas pyproj shapely folium mapclassify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 10.0 MB/s eta 0:00:00


## Configuration

Adjust `RESOLUTION_M` to control raster file size. At 10 m (native Sentinel-2 resolution for
these bands), the band arrays are roughly 64 MB each. At 20 m they're ~16 MB each. The choice
doesn't affect the MSOA-level analysis much — zonal stats average over many pixels anyway — but
it does affect the quality of true-colour visualisations and any sub-MSOA analysis participants
might attempt.

In [2]:
import os, json, io, zipfile, warnings, time
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.transform import Affine
from rasterio.enums import Resampling
from rasterio.warp import reproject as rio_reproject
from pyproj import Transformer
from shapely.geometry import box, Point, shape as shp_shape
from shapely.ops import unary_union
import pystac_client
import planetary_computer

# ======== CONFIGURATION ========
RESOLUTION_M = 20          # pixel size in metres (10, 20, 30, ...)
EARLY_YEAR = 2019
LATE_YEAR = 2024
MAX_CLOUD_COVER = 15       # percent

# London bounding box (WGS84)
LONDON_BBOX = [-0.51, 51.28, 0.33, 51.69]

# Charing Cross (lon, lat)
CHARING_CROSS = (-0.1276, 51.5074)

# Sentinel-2 bands to download
BANDS = ["B02", "B03", "B04", "B08"]
BAND_NAMES = {"B02": "Blue", "B03": "Green", "B04": "Red", "B08": "NIR"}
S2_SCALE = 0.0001  # L2A reflectance scale factor

# London boroughs (for filtering MSOAs)
LONDON_BOROUGHS = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Camden",
    "City of London", "Croydon", "Ealing", "Enfield", "Greenwich", "Hackney",
    "Hammersmith and Fulham", "Haringey", "Harrow", "Havering", "Hillingdon",
    "Hounslow", "Islington", "Kensington and Chelsea", "Kingston upon Thames",
    "Lambeth", "Lewisham", "Merton", "Newham", "Redbridge", "Richmond upon Thames",
    "Southwark", "Sutton", "Tower Hamlets", "Waltham Forest", "Wandsworth",
    "Westminster",
]

OUT_DIR = "geojam_data"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Resolution: {RESOLUTION_M} m")
print(f"Years: {EARLY_YEAR} and {LATE_YEAR}")
print(f"Output directory: {OUT_DIR}/")

Resolution: 20 m
Years: 2019 and 2024
Output directory: geojam_data/


## 1. Sentinel-2 imagery

We search Microsoft Planetary Computer's STAC catalogue for summer (June–August)
Sentinel-2 L2A scenes covering the London bounding box.

London spans multiple Sentinel-2 MGRS tiles — no single tile covers the whole city.
We group available scenes by date, compute the combined coverage of each date's tiles
over the London bbox, and pick the date with the best combination of coverage and low
cloud. We then mosaic all tiles from that date into a single raster in British National
Grid (EPSG:27700).

In [3]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def find_best_scenes(year):
    """Find a set of tiles from the same date that best cover the London bbox."""
    target = box(*LONDON_BBOX)

    for y in [year, year - 1, year + 1]:
        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=LONDON_BBOX,
            datetime=f"{y}-06-01/{y}-08-31",
            query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
            max_items=200,
        )
        items = list(search.items())
        if not items:
            continue

        # Group by date
        by_date = defaultdict(list)
        for item in items:
            by_date[item.datetime.strftime("%Y-%m-%d")].append(item)

        # Score each date: coverage minus cloud penalty
        best_date, best_score, best_items = None, -1, []
        for date_str, date_items in by_date.items():
            combined = unary_union([shp_shape(it.geometry) for it in date_items])
            coverage = combined.intersection(target).area / target.area
            avg_cloud = np.mean([it.properties["eo:cloud_cover"] for it in date_items])
            score = coverage - avg_cloud / 100
            if score > best_score:
                best_score = score
                best_date = date_str
                best_items = date_items

        if y != year:
            print(f"  No good scenes in {year}, using {y} instead")

        tiles = [it.properties.get("s2:mgrs_tile", "?") for it in best_items]
        clouds = [it.properties["eo:cloud_cover"] for it in best_items]
        combined = unary_union([shp_shape(it.geometry) for it in best_items])
        cov = combined.intersection(target).area / target.area

        print(f"  Date: {best_date}")
        print(f"  Tiles: {tiles}")
        print(f"  Cloud cover per tile: {[f'{c:.1f}%' for c in clouds]}")
        print(f"  Combined London coverage: {cov:.0%}")
        return best_items

    raise RuntimeError(f"No suitable scenes for {year} +/- 1 year")

print("Searching for early scenes...")
early_scenes = find_best_scenes(EARLY_YEAR)
print("\nSearching for late scenes...")
late_scenes = find_best_scenes(LATE_YEAR)

Searching for early scenes...
  Date: 2019-07-23
  Tiles: ['31UCT', '31UCS', '30UYC', '30UYB', '30UXC', '30UXB']
  Cloud cover per tile: ['0.1%', '0.1%', '0.8%', '0.1%', '4.5%', '0.1%']
  Combined London coverage: 99%

Searching for late scenes...
  Date: 2024-07-29
  Tiles: ['31UCT', '31UCS', '30UYC', '30UYB', '30UXC', '30UXB']
  Cloud cover per tile: ['3.0%', '0.0%', '2.7%', '0.0%', '7.2%', '0.1%']
  Combined London coverage: 100%


In [4]:
def read_and_mosaic(scenes, bbox_wgs84, resolution_m):
    """Read bands from multiple Sentinel-2 tiles, reproject to BNG, and mosaic."""
    dst_crs = "EPSG:27700"

    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    left, bottom = tf.transform(bbox_wgs84[0], bbox_wgs84[1])
    right, top = tf.transform(bbox_wgs84[2], bbox_wgs84[3])

    out_w = int((right - left) / resolution_m)
    out_h = int((top - bottom) / resolution_m)
    out_transform = Affine(resolution_m, 0, left, 0, -resolution_m, top)

    bands = {}
    meta = None

    for band_name in BANDS:
        mosaic = np.zeros((out_h, out_w), dtype=np.float32)
        count = np.zeros((out_h, out_w), dtype=np.float32)

        for scene in scenes:
            with rasterio.open(scene.assets[band_name].href) as src:
                tf_inv = Transformer.from_crs(dst_crs, src.crs, always_xy=True)
                sl, sb = tf_inv.transform(left, bottom)
                sr, st = tf_inv.transform(right, top)

                src_win = from_bounds(sl, sb, sr, st, src.transform)
                r0 = max(0, int(src_win.row_off))
                c0 = max(0, int(src_win.col_off))
                r1 = min(src.height, int(src_win.row_off + src_win.height))
                c1 = min(src.width, int(src_win.col_off + src_win.width))

                if r1 <= r0 or c1 <= c0:
                    continue

                win = Window(c0, r0, c1 - c0, r1 - r0)
                src_data = src.read(1, window=win).astype(np.float32) * S2_SCALE
                src_tf = rasterio.windows.transform(win, src.transform)

                tile = np.zeros((out_h, out_w), dtype=np.float32)
                rio_reproject(
                    source=src_data,
                    destination=tile,
                    src_transform=src_tf,
                    src_crs=src.crs,
                    dst_transform=out_transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.bilinear,
                    src_nodata=0,
                    dst_nodata=0,
                )
                valid = tile > 0
                mosaic[valid] += tile[valid]
                count[valid] += 1

        has_data = count > 0
        mosaic[has_data] /= count[has_data]
        bands[band_name] = mosaic

        if meta is None:
            meta = {
                "crs": dst_crs,
                "transform": list(out_transform)[:6],
                "shape": [out_h, out_w],
                "resolution_m": resolution_m,
                "bounds_wgs84": bbox_wgs84,
            }

        coverage = has_data.mean()
        vmin = mosaic[has_data].min() if has_data.any() else 0
        vmax = mosaic[has_data].max() if has_data.any() else 0
        print(f"  {band_name} ({BAND_NAMES[band_name]:>5}): {mosaic.shape}  "
              f"range [{vmin:.4f}, {vmax:.4f}]  coverage {coverage:.0%}")

    return bands, meta


print("Downloading and mosaicing early bands...")
early_bands, raster_meta = read_and_mosaic(early_scenes, LONDON_BBOX, RESOLUTION_M)

print("\nDownloading and mosaicing late bands...")
late_bands, _ = read_and_mosaic(late_scenes, LONDON_BBOX, RESOLUTION_M)

raster_meta["early_date"] = early_scenes[0].datetime.strftime("%Y-%m-%d")
raster_meta["late_date"] = late_scenes[0].datetime.strftime("%Y-%m-%d")
raster_meta["early_scene_ids"] = [s.id for s in early_scenes]
raster_meta["late_scene_ids"] = [s.id for s in late_scenes]
raster_meta["band_order"] = BANDS

print(f"\nRaster shape: {raster_meta['shape']}")
print(f"CRS: {raster_meta['crs']}")

  B02 ( Blue): (2355, 2856)  range [0.0176, 1.6019]  coverage 99%
  B03 (Green): (2355, 2856)  range [0.0134, 1.5260]  coverage 99%
  B04 (  Red): (2355, 2856)  range [0.0074, 1.5618]  coverage 99%
  B08 (  NIR): (2355, 2856)  range [0.0032, 1.5402]  coverage 99%

  B02 ( Blue): (2355, 2856)  range [0.1017, 1.6826]  coverage 99%
  B03 (Green): (2355, 2856)  range [0.1065, 1.6467]  coverage 99%
  B04 (  Red): (2355, 2856)  range [0.1011, 1.6703]  coverage 99%
  B08 (  NIR): (2355, 2856)  range [0.0983, 1.6498]  coverage 99%

Raster shape: [2355, 2856]
CRS: EPSG:27700


In [5]:
def compute_ndvi(bands):
    nir, red = bands["B08"], bands["B04"]
    denom = nir + red
    return np.where(denom > 0, (nir - red) / denom, 0.0).astype(np.float32)

ndvi_early = compute_ndvi(early_bands)
ndvi_late = compute_ndvi(late_bands)

print(f"NDVI early: range [{ndvi_early.min():.3f}, {ndvi_early.max():.3f}]")
print(f"NDVI late:  range [{ndvi_late.min():.3f}, {ndvi_late.max():.3f}]")

NDVI early: range [-0.849, 0.927]
NDVI late:  range [-0.510, 0.786]


## 2. MSOA boundaries

Middle layer Super Output Areas (MSOAs) are census geography units with roughly
5,000–15,000 residents each. We use the 2021 boundaries (BGC — generalised clipped)
from the ONS ArcGIS REST API.

The service name changes periodically when ONS republishes, so we include a fallback
that tries alternative endpoints. After download, we filter to London boroughs only.

In [6]:
def download_msoa_arcgis():
    """Paginated download from ONS ArcGIS REST API, spatially filtered to London."""
    service = "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3"
    base = (f"https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
            f"{service}/FeatureServer/0/query")

    w, s, e, n = LONDON_BBOX
    features, offset = [], 0

    while True:
        params = {
            "where": "1=1",
            "geometryType": "esriGeometryEnvelope",
            "geometry": f"{w},{s},{e},{n}",
            "inSR": 4326,
            "spatialRel": "esriSpatialRelIntersects",
            "outFields": "MSOA21CD,MSOA21NM",
            "returnGeometry": True,
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": 1000,
        }
        r = requests.get(base, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("features", [])
        if not batch:
            break
        features.extend(batch)
        print(f"  {len(features)} MSOAs downloaded...")
        if len(batch) < 1000:
            break
        offset += 1000

    gdf = gpd.GeoDataFrame.from_features(
        {"type": "FeatureCollection", "features": features}, crs="EPSG:4326"
    )
    return gdf


def download_msoa_fallback():
    """Fallback: try alternative service names and direct download."""
    alt_services = [
        "MSOA_2021_EW_BGC_V3",
        "Middle_layer_Super_Output_Areas_2021_EW_BGC_V3",
        "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC",
    ]
    for svc in alt_services:
        try:
            base = (f"https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
                    f"{svc}/FeatureServer/0/query")
            w, s, e, n = LONDON_BBOX
            params = {
                "where": "1=1",
                "geometry": f"{w},{s},{e},{n}",
                "geometryType": "esriGeometryEnvelope",
                "inSR": 4326,
                "spatialRel": "esriSpatialRelIntersects",
                "outFields": "MSOA21CD,MSOA21NM",
                "returnGeometry": True,
                "outSR": 4326,
                "f": "geojson",
                "resultRecordCount": 2000,
            }
            r = requests.get(base, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            if data.get("features"):
                print(f"  Fallback succeeded with service: {svc}")
                return gpd.GeoDataFrame.from_features(data, crs="EPSG:4326")
        except Exception:
            continue

    print("  Trying direct GeoJSON download (large file, may be slow)...")
    url = ("https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/"
           "4eba0534e6d842e5b1e1b0e3ab31b2a1/geojson")
    gdf = gpd.read_file(url)
    london_box = box(*LONDON_BBOX)
    return gdf[gdf.intersects(london_box)].copy()


print("Downloading MSOA 2021 boundaries...")
try:
    msoa = download_msoa_arcgis()
except Exception as exc:
    print(f"  Primary API failed: {exc}")
    print("  Trying fallbacks...")
    msoa = download_msoa_fallback()

# Keep only English MSOAs
msoa = msoa[msoa["MSOA21CD"].str.startswith("E02")].copy()

# Filter to London boroughs only
def is_london_msoa(name):
    name_lower = name.lower()
    return any(name_lower.startswith(b.lower()) for b in LONDON_BOROUGHS)

before = len(msoa)
msoa = msoa[msoa["MSOA21NM"].apply(is_london_msoa)].copy()
msoa = msoa.reset_index(drop=True)
print(f"\n{before} MSOAs from bbox, {len(msoa)} after London borough filter")

  1000 MSOAs downloaded...
  1201 MSOAs downloaded...

1201 MSOAs from bbox, 1011 after London borough filter


## 3. Census population

Population counts from Census 2021, downloaded from the NOMIS bulk zip. The API
endpoint can return empty CSVs depending on the time of day, so the bulk zip is
more reliable. We detect column names dynamically because they vary between downloads.

In [7]:
def download_census_population():
    url = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip"
    print("  Downloading NOMIS bulk zip...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        csvs = [n for n in zf.namelist() if n.endswith(".csv")]
        csvs.sort(key=lambda n: (0 if "msoa" in n.lower() else 1, n))

        for name in csvs:
            df = pd.read_csv(zf.open(name))
            df.columns = df.columns.str.strip()

            geo_col = next(
                (c for c in df.columns
                 if "geography code" in c.lower() or c.lower() == "mnemonic"),
                None
            )
            if geo_col is None:
                continue

            mask = df[geo_col].astype(str).str.startswith("E02")
            if mask.sum() == 0:
                continue

            df = df[mask].copy()

            pop_col = next(
                (c for c in df.columns if "observation" in c.lower()),
                None
            )
            if pop_col is None:
                num_cols = df.select_dtypes(include=[np.number]).columns
                pop_col = num_cols[-1] if len(num_cols) > 0 else None

            if pop_col:
                out = df[[geo_col, pop_col]].copy()
                out.columns = ["MSOA21CD", "population"]
                out["population"] = pd.to_numeric(out["population"], errors="coerce")
                print(f"  Extracted from {name}: {len(out)} rows")
                return out

    raise RuntimeError("Could not extract MSOA population from NOMIS zip")


print("Downloading Census 2021 population...")
try:
    pop_df = download_census_population()
    msoa = msoa.merge(pop_df, on="MSOA21CD", how="left")
    matched = msoa["population"].notna().sum()
    print(f"  Matched {matched}/{len(msoa)} MSOAs")
except Exception as exc:
    print(f"  Census download failed: {exc}")
    print("  Using area-based proxy (rough approximation)")
    msoa_proj = msoa.to_crs("EPSG:27700")
    msoa["population"] = (msoa_proj.geometry.area / 1e6 * 5600).astype(int)

  Extracted from census2021-ts001-msoa.csv: 6856 rows
  Matched 1011/1011 MSOAs


## 4. Spatial features

These are features derived from geometry and census data — no satellite imagery involved.
Participants receive these pre-computed so they can focus on the satellite-derived features
and ML during the hackathon.

In [8]:
# Project to British National Grid for distance/area calculations
msoa_bng = msoa.to_crs("EPSG:27700")

# Area
msoa["area_km2"] = msoa_bng.geometry.area / 1e6

# Population density
msoa["pop_density"] = msoa["population"] / msoa["area_km2"]

# Distance to Charing Cross
cc = gpd.GeoSeries([Point(CHARING_CROSS)], crs="EPSG:4326").to_crs("EPSG:27700").iloc[0]
msoa["dist_to_centre_km"] = msoa_bng.geometry.centroid.distance(cc) / 1000

print(f"Area: {msoa['area_km2'].min():.2f} – {msoa['area_km2'].max():.2f} km²")
print(f"Density: {msoa['pop_density'].min():.0f} – {msoa['pop_density'].max():.0f} per km²")
print(f"Distance to centre: {msoa['dist_to_centre_km'].min():.1f} – {msoa['dist_to_centre_km'].max():.1f} km")

Area: 0.29 – 47.81 km²
Density: 0 – 3545 per km²
Distance to centre: 0.4 – 38.7 km


## 5. OSM parks

We download park polygons from OpenStreetMap and compute the distance from each MSOA
centroid to the nearest park boundary. This is a proxy for green space access.

The download covers all of Greater London and may take a few minutes.

In [9]:
import osmnx as ox

def download_parks():
    tags = {"leisure": "park"}
    try:
        return ox.features_from_place("Greater London, England", tags=tags)
    except AttributeError:
        pass
    try:
        return ox.geometries_from_place("Greater London, England", tags=tags)
    except Exception:
        pass
    w, s, e, n = LONDON_BBOX
    try:
        return ox.features_from_bbox(bbox=(n, s, e, w), tags=tags)
    except TypeError:
        return ox.features_from_bbox(n, s, e, w, tags=tags)


print("Downloading parks from OSM...")
try:
    parks = download_parks()
    parks = parks[parks.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    parks_bng = parks.to_crs("EPSG:27700")
    print(f"  {len(parks)} park polygons")

    park_union = unary_union(parks_bng.geometry.values)
    centroids_bng = msoa_bng.geometry.centroid
    msoa["dist_to_park_km"] = centroids_bng.distance(park_union) / 1000

    print(f"  Distance to park: {msoa['dist_to_park_km'].min():.2f} – "
          f"{msoa['dist_to_park_km'].max():.2f} km")
except Exception as exc:
    print(f"  Parks download failed: {exc}")
    msoa["dist_to_park_km"] = np.nan
    print("  dist_to_park_km set to NaN")

  3236 park polygons
  Distance to park: 0.00 – 10.11 km


## 6. Escape hatch: zonal statistics

During the hackathon, participants will compute zonal statistics themselves — aggregating
NDVI raster values into each MSOA polygon. That's the core spatial task.

But if a team gets stuck on CRS mismatches or rasterstats issues, they need a way forward.
Here we pre-compute the satellite-derived features and save a "full" GeoDataFrame that
includes everything. In Notebook 2 this is behind a clearly marked bailout cell.

In [10]:
from rasterstats import zonal_stats

# Reconstruct affine transform for rasterstats
tf_values = raster_meta["transform"]
aff = Affine(*tf_values)
raster_crs = raster_meta["crs"]

# Reproject MSOAs to match raster CRS for zonal stats
msoa_raster_crs = msoa.to_crs(raster_crs)

print("Computing zonal statistics (this takes a minute or two)...")

def zstats(raster, stat="mean"):
    results = zonal_stats(
        msoa_raster_crs.geometry, raster,
        affine=aff, stats=[stat], nodata=0.0
    )
    return [r[stat] for r in results]

msoa_full = msoa.copy()
msoa_full["mean_ndvi_early"] = zstats(ndvi_early)
print("  mean_ndvi_early done")

msoa_full["mean_ndvi_late"] = zstats(ndvi_late)
print("  mean_ndvi_late done")

delta_ndvi = ndvi_late - ndvi_early
msoa_full["mean_delta_ndvi"] = zstats(delta_ndvi)
print("  mean_delta_ndvi done")

# Proportion of pixels that got greener vs greyer
greener = (delta_ndvi > 0).astype(np.float32)
msoa_full["prop_greener"] = zstats(greener)
msoa_full["prop_greyer"] = 1.0 - msoa_full["prop_greener"]
print("  proportions done")

# Flag MSOAs with no raster coverage
no_coverage = msoa_full["mean_ndvi_early"].isna()
print(f"\n{no_coverage.sum()} MSOAs have no raster coverage")
if no_coverage.sum() > 0:
    print(f"  Examples: {msoa_full.loc[no_coverage, 'MSOA21NM'].head(5).tolist()}")

Computing zonal statistics (this takes a minute or two)...
  mean_ndvi_early done
  mean_ndvi_late done
  mean_delta_ndvi done
  proportions done

1 MSOAs have no raster coverage
  Examples: ['Brentwood 003']


## 7. Save outputs

In [11]:
# --- Raster bands ---
for band_name in BANDS:
    np.save(os.path.join(OUT_DIR, f"band_early_{band_name}.npy"), early_bands[band_name])
    np.save(os.path.join(OUT_DIR, f"band_late_{band_name}.npy"), late_bands[band_name])
print(f"Saved {len(BANDS) * 2} band arrays")

# --- NDVI rasters ---
np.save(os.path.join(OUT_DIR, "ndvi_early.npy"), ndvi_early)
np.save(os.path.join(OUT_DIR, "ndvi_late.npy"), ndvi_late)
print("Saved NDVI rasters")

# --- Raster metadata ---
with open(os.path.join(OUT_DIR, "raster_meta.json"), "w") as f:
    json.dump(raster_meta, f, indent=2)
print("Saved raster_meta.json")

# --- Base MSOA GeoDataFrame (non-satellite features only) ---
msoa.to_file(os.path.join(OUT_DIR, "msoa_base.gpkg"), driver="GPKG")
print(f"Saved msoa_base.gpkg ({len(msoa)} MSOAs, {len(msoa.columns)} columns)")
print(f"  Columns: {list(msoa.columns)}")

# --- Full MSOA GeoDataFrame (escape hatch) ---
msoa_full.to_file(os.path.join(OUT_DIR, "msoa_full.gpkg"), driver="GPKG")
print(f"Saved msoa_full.gpkg ({len(msoa_full)} MSOAs, {len(msoa_full.columns)} columns)")
print(f"  Additional columns: {[c for c in msoa_full.columns if c not in msoa.columns]}")

Saved 8 band arrays
Saved NDVI rasters
Saved raster_meta.json
Saved msoa_base.gpkg (1011 MSOAs, 8 columns)
  Columns: ['geometry', 'MSOA21CD', 'MSOA21NM', 'population', 'area_km2', 'pop_density', 'dist_to_centre_km', 'dist_to_park_km']
Saved msoa_full.gpkg (1011 MSOAs, 13 columns)
  Additional columns: ['mean_ndvi_early', 'mean_ndvi_late', 'mean_delta_ndvi', 'prop_greener', 'prop_greyer']


In [12]:
# --- README ---
early_ids = '\n  '.join(raster_meta['early_scene_ids'])
late_ids = '\n  '.join(raster_meta['late_scene_ids'])
base_cols = ', '.join(c for c in msoa.columns if c != 'geometry')
extra_cols = ', '.join(c for c in msoa_full.columns if c not in msoa.columns)

readme = f"""GeoJam 2026 — Data Pack
========================

Prepared by the data prep notebook on {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}.

Satellite imagery
-----------------
Source: Sentinel-2 L2A via Microsoft Planetary Computer
Early scenes ({raster_meta['early_date']}):
  {early_ids}
Late scenes ({raster_meta['late_date']}):
  {late_ids}
Resolution:  {raster_meta['resolution_m']} m/pixel
Raster shape: {raster_meta['shape'][0]} x {raster_meta['shape'][1]} pixels
CRS: {raster_meta['crs']}

Files:
  band_early_B02.npy .. B08.npy  — Sentinel-2 bands (float32, reflectance 0-1)
  band_late_B02.npy  .. B08.npy  — same for the later year
  ndvi_early.npy, ndvi_late.npy  — pre-computed NDVI (float32, range -1 to 1)
  raster_meta.json               — affine transform, CRS, shape, scene IDs

  Bands: B02=Blue, B03=Green, B04=Red, B08=NIR
  True colour composite: RGB = B04, B03, B02
  NDVI = (B08 - B04) / (B08 + B04)

MSOA data
---------
  msoa_base.gpkg  — {len(msoa)} London MSOAs with non-satellite features:
    {base_cols}

  msoa_full.gpkg  — same MSOAs with satellite-derived features added (escape hatch):
    Additional: {extra_cols}

Sources
-------
  Boundaries: ONS MSOA 2021 (BGC)
  Population: Census 2021 via NOMIS (TS001)
  Parks: OpenStreetMap (leisure=park)

Caveats
-------
  - Mosaiced from multiple Sentinel-2 tiles. Some edge MSOAs may still lack
    raster coverage ({no_coverage.sum()} MSOAs had no data in this run).
  - NDVI from a single scene is affected by weather, phenology, and atmospheric
    conditions on that specific day. It is not a multi-temporal average.
  - Park data from OSM varies in completeness.
"""

with open(os.path.join(OUT_DIR, "README.txt"), "w") as f:
    f.write(readme)
print("Saved README.txt")

Saved README.txt


In [13]:
import shutil

zip_path = "geojam_data.zip"
shutil.make_archive("geojam_data", "zip", ".", OUT_DIR)

zip_size_mb = os.path.getsize(zip_path) / 1e6
print(f"Created {zip_path} ({zip_size_mb:.1f} MB)")

# List contents
print("\nContents:")
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    sz = os.path.getsize(fpath) / 1e6
    print(f"  {fname:40s} {sz:8.1f} MB")

Created geojam_data.zip (232.9 MB)

Contents:
  README.txt                                    0.0 MB
  band_early_B02.npy                           26.9 MB
  band_early_B03.npy                           26.9 MB
  band_early_B04.npy                           26.9 MB
  band_early_B08.npy                           26.9 MB
  band_late_B02.npy                            26.9 MB
  band_late_B03.npy                            26.9 MB
  band_late_B04.npy                            26.9 MB
  band_late_B08.npy                            26.9 MB
  msoa_base.gpkg                                1.8 MB
  msoa_full.gpkg                                1.8 MB
  ndvi_early.npy                               26.9 MB
  ndvi_late.npy                                26.9 MB
  raster_meta.json                              0.0 MB


In [14]:
# Download the zip in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("Download started. Check your browser downloads.")
except ImportError:
    print(f"Not running in Colab. Zip is at: {os.path.abspath(zip_path)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Check your browser downloads.


## Summary

In [15]:
print("=" * 60)
print("GeoJam Data Prep v2 — Summary")
print("=" * 60)
print(f"Early scenes: {raster_meta['early_scene_ids']}")
print(f"              {raster_meta['early_date']}")
print(f"Late scenes:  {raster_meta['late_scene_ids']}")
print(f"              {raster_meta['late_date']}")
print(f"Resolution:   {raster_meta['resolution_m']} m")
print(f"Raster shape: {raster_meta['shape']}")
print(f"MSOAs:        {len(msoa)} (base) / {len(msoa_full)} (full)")
print(f"No coverage:  {no_coverage.sum()} MSOAs")
print(f"Zip size:     {zip_size_mb:.1f} MB")
print()
print("Base GeoDataFrame columns:")
for c in msoa.columns:
    if c != "geometry":
        print(f"  {c}")
print()
print("Full GeoDataFrame adds:")
for c in msoa_full.columns:
    if c not in msoa.columns:
        print(f"  {c}")
print("=" * 60)

GeoJam Data Prep v2 — Summary
Early scenes: ['S2B_MSIL2A_20190723T105629_R094_T31UCT_20201106T013939', 'S2B_MSIL2A_20190723T105629_R094_T31UCS_20201005T184044', 'S2B_MSIL2A_20190723T105629_R094_T30UYC_20201005T184045', 'S2B_MSIL2A_20190723T105629_R094_T30UYB_20201005T184040', 'S2B_MSIL2A_20190723T105629_R094_T30UXC_20201005T184035', 'S2B_MSIL2A_20190723T105629_R094_T30UXB_20201005T184037']
              2019-07-23
Late scenes:  ['S2B_MSIL2A_20240729T110619_R137_T31UCT_20240729T135102', 'S2B_MSIL2A_20240729T110619_R137_T31UCS_20240729T135102', 'S2B_MSIL2A_20240729T110619_R137_T30UYC_20240729T135102', 'S2B_MSIL2A_20240729T110619_R137_T30UYB_20240729T135102', 'S2B_MSIL2A_20240729T110619_R137_T30UXC_20240729T135102', 'S2B_MSIL2A_20240729T110619_R137_T30UXB_20240729T135102']
              2024-07-29
Resolution:   20 m
Raster shape: [2355, 2856]
MSOAs:        1011 (base) / 1011 (full)
No coverage:  1 MSOAs
Zip size:     232.9 MB

Base GeoDataFrame columns:
  MSOA21CD
  MSOA21NM
  population
